# MPMWorlds SFT 划分：数据泄漏审查

对已经划分好 train / val / test 的数据做泄漏审查。**只读，不会改动任何数据文件。**

审查的泄漏类型（从严重到轻）：

| # | 检查 | 为什么重要 |
|---|---|---|
| 1 | **家族泄漏**：同一个 base 场景的 perturbation 兄弟分到了不同 split | 最严重，评估会严重虚高 |
| 2 | **精确重复**：yaml / program / video 内容完全相同 | 直接拷贝 |
| 3 | **程序近重复**：测试程序在训练集里有 >0.95 相似的拷贝 | 模型可能靠记忆代码 |
| 4 | **视频近重复**：初始帧/关键帧几乎相同 | 视觉上记忆 |
| 5 | **参数/结构覆盖**：评估集的参数组合、结构、材料在训练集里的出现情况，以及"猜众数"基线 | 判断指标的下限 |
| 6 | **提示侧泄漏**：路径、场景名、description 等是否会漏进 prompt | 构建 SFT 数据时必须避开 |

最后一个 cell 汇总成 PASS / WARN / FAIL 表并保存报告。

In [ ]:
from pathlib import Path
import re, json, hashlib, collections, math, random
import numpy as np, pandas as pd
import yaml
import matplotlib.pyplot as plt

# ======================= 配置：改这里 =======================
DATA_ROOT  = Path("/Users/ruowen_vagabond/Projects-26fall/pixels2physics/data/SFT-2000/raw")
LAYOUT     = "dir"          # "dir": 每个场景一个目录（含 scene.yaml/program.py/video.mp4）；"flat": xxx.yaml / xxx.py / xxx.mp4 同名配对
YAML_NAME, PROG_NAME, VIDEO_NAME = "scene.yaml", "program.py", "video.mp4"

# split 的来源：默认从路径里识别 train/val/test（目录名或目录名中的单词）。
# 如果你的划分是名单文件，填这里（每行一个场景 id 或目录名），优先级高于路径识别：
SPLIT_FILES = {}            # 例: {"train": "train.txt", "val": "val.txt", "test": "test.txt"}
SPLIT_ALIASES = {"train": "train", "training": "train", "val": "val", "valid": "val", "validation": "val",
                 "dev": "val", "test": "test", "testing": "test", "eval": "test"}

CHECK_VIDEO   = True
VIDEO_FRAC    = (0.0, 0.5, 0.99)     # 取视频的哪些相对位置的帧做签名
VIDEO_RMS_THR = 0.02                 # 32x32 灰度签名的 RMS 差（0-1）低于此值视为视觉近重复
PROG_SIM_THR  = 0.95                 # 程序 TF-IDF 余弦相似度阈值
WARN_FAIL     = dict(prog=(0.05, 0.30), video=(0.05, 0.30))   # 近重复占评估集比例的 (WARN, FAIL) 阈值
OUT_DIR = Path("audit_out"); OUT_DIR.mkdir(exist_ok=True)
# ============================================================
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.width", 200, "display.max_columns", 50, "display.max_colwidth", 80)
AUDIT = []
def audit(check, value, status, note=""): AUDIT.append(dict(check=check, value=value, status=status, note=note))
def grade(v, warn, fail): return "FAIL" if v > fail else ("WARN" if v > warn else "PASS")

## 1. 载入、识别 split、清点

In [ ]:
def to_num(x):
    if isinstance(x, bool): return np.nan
    if isinstance(x, (int, float)): return float(x)
    if isinstance(x, str):
        try: return float(x)
        except ValueError: return np.nan
    return np.nan
def as_dict(x): return x if isinstance(x, dict) else {}
def get_objects(spec):
    o = spec.get("objects")
    if isinstance(o, dict): return [dict(v, name=k) if isinstance(v, dict) else {"name": k} for k, v in o.items()]
    if isinstance(o, list): return [x if isinstance(x, dict) else {"name": str(x)} for x in o]
    return []
def obj_sig(o):
    pp = as_dict(o.get("physics_properties"))
    shp = o.get("shape") or as_dict(o.get("initial_state")).get("shape") or {}
    return str((o.get("object_type"), shp if isinstance(shp, str) else as_dict(shp).get("type"), pp.get("constitutive_model")))
def scene_sig(spec): return " | ".join(sorted(obj_sig(o) for o in get_objects(spec)))
def prim_pp(spec):
    for o in get_objects(spec):
        pp = as_dict(o.get("physics_properties"))
        if pp.get("constitutive_model") is not None: return pp
    return {}

def split_of(rel):
    for part in Path(rel).parts[:-1]:
        for tok in re.split(r"[^a-z]+", part.lower()):
            if tok in SPLIT_ALIASES: return SPLIT_ALIASES[tok]
    return None

_fam_re = re.compile(r"^(?P<fam>.*?scene_\d+)_(?:base|perturbation_\d+)$")
def family_key(sid):
    base = Path(sid).name
    m = _fam_re.match(base)
    fam = m.group("fam") if m else base
    p = re.search(r"part_\d+", sid)          # 若场景编号在各 part 内重复，加上 part 前缀避免误判
    return (p.group(0) + "/" if p else "") + fam
def kind_of(sid):
    b = Path(sid).name
    return "base" if b.endswith("_base") else ("perturbation" if "_perturbation_" in b else "other")
md5 = lambda b: hashlib.md5(b).hexdigest()
def template(code):
    c = re.sub(r"#.*", "", code); c = re.sub(r"\b\d+\.?\d*(?:[eE][-+]?\d+)?\b", "N", c)
    return md5(re.sub(r"\s+", " ", c).encode())[:12]

print("DATA_ROOT:", DATA_ROOT, "| exists:", DATA_ROOT.exists())
if LAYOUT == "dir":
    items = [(str(p.parent.relative_to(DATA_ROOT)), p, p.parent / PROG_NAME, p.parent / VIDEO_NAME) for p in sorted(DATA_ROOT.rglob(YAML_NAME))]
else:
    ys = sorted(list(DATA_ROOT.rglob("*.yaml")) + list(DATA_ROOT.rglob("*.yml")))
    items = [(str(p.relative_to(DATA_ROOT).with_suffix("")), p, p.with_suffix(".py"), p.with_suffix(".mp4")) for p in ys]
print(f"found {len(items)} scenes")
if not items:
    files = [p for p in DATA_ROOT.rglob("*") if p.is_file()] if DATA_ROOT.exists() else []
    print(f"{len(files)} files; suffixes:", collections.Counter(p.suffix for p in files).most_common(10))
    for p in files[:15]: print("  ", p.relative_to(DATA_ROOT))
    raise FileNotFoundError("没找到场景：检查 DATA_ROOT / LAYOUT / YAML_NAME")

file_split = {}
for k, f in SPLIT_FILES.items():
    for line in Path(f).read_text().split(): file_split[line.strip()] = SPLIT_ALIASES.get(k.lower(), k.lower())

rows, progs, problems = [], {}, []
for sid, yp, pp_, vp in items:
    try:
        spec = yaml.safe_load(yp.read_text()); assert isinstance(spec, dict)
    except Exception as e:
        problems.append((sid, "yaml_parse", str(e)[:80])); continue
    code_txt = pp_.read_text(errors="ignore") if pp_.exists() else None
    if code_txt is None: problems.append((sid, "missing_program", ""))
    if not vp.exists(): problems.append((sid, "missing_video", ""))
    if code_txt is not None: progs[sid] = code_txt
    pp = prim_pp(spec); sim = as_dict(spec.get("simulation"))
    sp = file_split.get(sid) or file_split.get(Path(sid).name) if file_split else split_of(sid)
    rows.append(dict(scene=sid, split=sp or "unknown", family=family_key(sid), kind=kind_of(sid),
        cm=str(pp.get("constitutive_model")), rho=to_num(pp.get("density")), E=to_num(pp.get("youngs_modulus")), nu=to_num(pp.get("poissons_ratio")),
        dt=to_num(sim.get("time_step_dt")), substeps=to_num(sim.get("substeps_per_frame")),
        sig=scene_sig(spec), n_objects=len(get_objects(spec)),
        yaml_hash=md5(json.dumps(spec, sort_keys=True, default=str).encode()),
        prog_hash=md5(re.sub(r"\s+", " ", code_txt).encode()) if code_txt else None,
        tmpl=template(code_txt) if code_txt else None, vpath=str(vp), has_video=vp.exists(),
        has_desc=bool(spec.get("description"))))
meta = pd.DataFrame(rows)
meta["sim_dt_per_frame"] = meta.dt * meta.substeps
meta["logE"], meta["logrho"] = np.log10(meta.E.where(meta.E > 0)), np.log10(meta.rho.where(meta.rho > 0))
meta["logE_over_rho"] = meta.logE - meta.logrho

if problems: display(pd.DataFrame(problems, columns=["scene", "type", "detail"]).groupby("type").size().rename("count").to_frame())
if (meta.split == "unknown").any():
    print(f"!! {(meta.split=='unknown').sum()} 个场景无法识别 split。路径示例：")
    for s in meta[meta.split == "unknown"].scene.head(8): print("   ", s)
    if (meta.split == "unknown").all():
        raise RuntimeError("所有场景都没识别出 split：请设置 SPLIT_FILES，或检查路径里是否含 train/val/test")
SPLITS = sorted(set(meta.split) - {"unknown"}); EVAL = [s for s in SPLITS if s != "train"]
print("splits:", SPLITS, "| eval splits:", EVAL)
display(pd.crosstab(meta.split, meta.kind, margins=True))
display(meta.groupby("split").agg(scenes=("scene", "count"), families=("family", "nunique")))
audit("split 规模", {s: int((meta.split == s).sum()) for s in SPLITS}, "WARN" if any((meta.split == s).sum() < 100 for s in EVAL) else "PASS", "评估集少于 100 条时指标方差很大")

## 2. 家族泄漏（最重要）

同一个 `scene_xxxxxx` 的 base 与所有 perturbation 必须在同一个 split。
家族 id 取自场景目录名（去掉 `_base` / `_perturbation_xxxxxx`），与所在的 train/val/test 目录无关。

In [ ]:
fs = meta.groupby("family").split.agg(lambda s: sorted(set(s)))
leaky = fs[fs.map(len) > 1]
ev_scenes = meta[meta.split.isin(EVAL)]
n_leaky_ev = int(ev_scenes.family.isin(leaky.index).sum())
print(f"家族总数 {len(fs)}；跨多个 split 的家族 {len(leaky)}；受影响的评估场景 {n_leaky_ev} / {len(ev_scenes)} ({n_leaky_ev/max(1,len(ev_scenes)):.1%})")
if len(leaky):
    off = meta[meta.family.isin(leaky.index)].groupby(["family", "split"]).size().unstack(fill_value=0)
    display(off.head(40)); off.to_csv(OUT_DIR / "offending_families.csv")
audit("家族泄漏：跨 split 的家族数", int(len(leaky)), "FAIL" if len(leaky) else "PASS", f"受影响评估场景 {n_leaky_ev}；修复：按家族 id 重新划分")
# base / perturbation 在各 split 的比例是否一致
kt = pd.crosstab(meta.split, meta.kind, normalize="index").round(3); display(kt)

## 3. 精确重复（跨 split）

In [ ]:
def cross_dup(col, label, fail=True):
    d = meta.dropna(subset=[col]); tr = set(d[d.split == "train"][col]); ev = d[d.split.isin(EVAL)]
    hit = ev[ev[col].isin(tr)]
    share = len(hit) / max(1, len(ev))
    print(f"{label:32s} 评估场景中在训练集有完全相同项: {len(hit):5d} / {len(ev)} ({share:.1%})")
    audit(f"精确重复：{label}", f"{len(hit)}/{len(ev)}", ("FAIL" if (fail and len(hit)) else ("WARN" if len(hit) else "PASS")))
    return hit
h1 = cross_dup("yaml_hash", "scene.yaml 完全相同")
h2 = cross_dup("prog_hash", "program.py 完全相同")
h3 = cross_dup("tmpl", "程序模板（数字归一化）相同", fail=False)
h4 = cross_dup("sig", "结构签名相同", fail=False)
for nm, h in [("yaml", h1), ("prog", h2)]:
    if len(h): display(h[["scene", "split", "family"]].head(10))
# >>> 看什么：yaml/program 完全相同必须为 0。模板相同只是提示（家族之间共用同一代码写法时会出现）；结构签名相同很常见，不算泄漏

## 4. 程序近重复（TF-IDF 余弦相似度，评估集 → 训练集）

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
ids = [s for s in meta.scene if s in progs]
spl = meta.set_index("scene").split.to_dict(); fam = meta.set_index("scene").family.to_dict()
clean = [re.sub(r"#.*", "", progs[i]) for i in ids]
vec = TfidfVectorizer(token_pattern=r"[A-Za-z_][A-Za-z_0-9]*", ngram_range=(1, 3), min_df=2, max_features=100000, sublinear_tf=True)
X = vec.fit_transform(clean)
tr_idx = [k for k, s in enumerate(ids) if spl[s] == "train"]; ev_idx = [k for k, s in enumerate(ids) if spl[s] in EVAL]
sims, arg = [], []
for st in range(0, len(ev_idx), 400):
    S = (X[ev_idx[st:st+400]] @ X[tr_idx].T).toarray(); sims.append(S.max(1)); arg.append(S.argmax(1))
sims, arg = np.concatenate(sims), np.concatenate(arg)
pairs = pd.DataFrame({"eval_scene": [ids[k] for k in ev_idx], "split": [spl[ids[k]] for k in ev_idx],
                      "train_neighbor": [ids[tr_idx[a]] for a in arg], "sim": sims})
pairs["same_family"] = [fam[a] == fam[b] for a, b in zip(pairs.eval_scene, pairs.train_neighbor)]
print("评估集 → 训练集 最近邻相似度：", pd.Series(sims).describe(percentiles=[.1, .5, .9, .99]).round(3).to_dict())
share = float((sims > PROG_SIM_THR).mean())
print(f"相似度 > {PROG_SIM_THR}: {share:.1%}；其中最近邻属于同一家族的占 {pairs[pairs.sim > PROG_SIM_THR].same_family.mean() if (sims > PROG_SIM_THR).any() else 0:.1%}")
audit(f"程序近重复（相似度>{PROG_SIM_THR}）占评估集比例", f"{share:.1%}", grade(share, *WARN_FAIL["prog"]), "若已按家族划分仍很高，说明不同家族共用同一套代码；评估时程序复制能力会被高估")
pairs.sort_values("sim", ascending=False).head(25).to_csv(OUT_DIR / "top_program_pairs.csv", index=False)
display(pairs.sort_values("sim", ascending=False).head(15))
fig, ax = plt.subplots(figsize=(6, 3)); ax.hist(np.clip(sims, 0, 1), bins=np.linspace(0, 1, 41)); ax.set_title("eval→train program similarity"); plt.show()

## 5. 视频近重复（关键帧缩略图签名）

In [ ]:
try:
    import cv2
except ImportError:
    cv2 = None; print("cv2 未安装：%pip install opencv-python-headless ；本节将跳过")

def video_sig(path):
    cap = cv2.VideoCapture(str(path)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n <= 0: cap.release(); return None
    out = []
    for fr in VIDEO_FRAC:
        cap.set(cv2.CAP_PROP_POS_FRAMES, min(n - 1, int(fr * n))); ok, img = cap.read()
        if not ok: cap.release(); return None
        out.append(cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (32, 32), interpolation=cv2.INTER_AREA))
    cap.release(); return np.stack(out)

def file_md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

if CHECK_VIDEO and cv2 is not None:
    sig_rows, V = [], []
    vs = meta[meta.has_video]
    for k, (sid, vp) in enumerate(zip(vs.scene, vs.vpath)):
        s = video_sig(vp)
        if s is None: continue
        sig_rows.append(dict(scene=sid, first_hash=md5(s[0].tobytes()), file_md5=file_md5(vp))); V.append(s.reshape(-1).astype(np.float32) / 255.0)
        if (k + 1) % 500 == 0: print(f"  {k+1}/{len(vs)}")
    vm = pd.DataFrame(sig_rows); V = np.stack(V)
    vm = vm.merge(meta[["scene", "split", "family"]], on="scene")
    print(f"视频签名完成：{len(vm)} / {len(vs)}")
    for col, label in [("file_md5", "视频文件完全相同"), ("first_hash", "首帧缩略图完全相同")]:
        tr = set(vm[vm.split == "train"][col]); ev = vm[vm.split.isin(EVAL)]; hit = ev[ev[col].isin(tr)]
        print(f"{label:20s}: {len(hit)} / {len(ev)} ({len(hit)/max(1,len(ev)):.1%})")
        audit(f"视频：{label}", f"{len(hit)}/{len(ev)}", ("FAIL" if (col == "file_md5" and len(hit)) else ("WARN" if len(hit) else "PASS")),
              "首帧相同通常意味着初始布局相同（同一家族的兄弟）" if col == "first_hash" else "")
    tr_m = (vm.split == "train").to_numpy(); ev_m = vm.split.isin(EVAL).to_numpy()
    A, B = V[ev_m], V[tr_m]; dim = V.shape[1]
    d2 = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    rms = np.sqrt(np.clip(d2, 0, None) / dim); nn = rms.argmin(1); nnd = rms.min(1)
    ev_ids, tr_ids = vm[ev_m].scene.to_numpy(), vm[tr_m].scene.to_numpy()
    vp_ = pd.DataFrame({"eval_scene": ev_ids, "train_neighbor": tr_ids[nn], "rms": nnd})
    vp_["same_family"] = [fam[a] == fam[b] for a, b in zip(vp_.eval_scene, vp_.train_neighbor)]
    share_v = float((nnd < VIDEO_RMS_THR).mean())
    print(f"最近邻 RMS 差 < {VIDEO_RMS_THR}: {share_v:.1%}；这些最近邻属于同一家族的占 {vp_[vp_.rms < VIDEO_RMS_THR].same_family.mean() if (nnd < VIDEO_RMS_THR).any() else 0:.1%}")
    audit(f"视频近重复（RMS<{VIDEO_RMS_THR}）占评估集比例", f"{share_v:.1%}", grade(share_v, *WARN_FAIL["video"]), "32x32 灰度、3 个关键帧；阈值是经验值，建议抽几对肉眼确认")
    vp_.sort_values("rms").head(25).to_csv(OUT_DIR / "video_pairs.csv", index=False); display(vp_.sort_values("rms").head(10))
else:
    print("跳过视频检查")

## 6. 参数 / 结构 / 材料覆盖，以及"猜众数"基线

不是泄漏，但决定指标怎么读：评估集的参数组合在训练集里出现得越多、众数基线越高，模型"不看视频"也能拿高分。

In [ ]:
TR, EV = meta[meta.split == "train"], meta[meta.split.isin(EVAL)]
unseen_cm = sorted(set(EV.cm) - set(TR.cm))
print("评估集有、训练集没有的材料:", unseen_cm or "无")
audit("评估集出现训练集没有的材料", unseen_cm or "无", "WARN" if unseen_cm else "PASS", "材料外推是难度，不是泄漏；但要知道指标里包含这部分")
tup = lambda d: set(zip(d.cm, d.rho.round(9).fillna(-1), d.E.round(9).fillna(-1), d.nu.round(9).fillna(-1)))
trt = tup(TR); seen = np.mean([t in trt for t in zip(EV.cm, EV.rho.round(9).fillna(-1), EV.E.round(9).fillna(-1), EV.nu.round(9).fillna(-1))])
sig_seen = EV.sig.isin(set(TR.sig)).mean()
print(f"评估场景的 (材料, ρ, E, ν) 组合在训练集出现过: {seen:.1%}；结构签名出现过: {sig_seen:.1%}")
audit("评估集参数组合在训练集出现过", f"{seen:.1%}", "PASS", "仅供参考：该值高说明参数多样性低，结合下面众数基线看")
audit("评估集结构签名在训练集出现过", f"{sig_seen:.1%}", "PASS", "仅供参考：低表示结构泛化更难")

rows = []
for cm, sub in TR.groupby("cm"):
    ev = EV[EV.cm == cm]
    if ev.empty: continue
    for p in ["rho", "E", "nu"]:
        mode = sub[p].round(9).dropna().mode()
        if mode.empty: continue
        rows.append(dict(cm=cm, param=p, n_eval=len(ev), train_mode=mode.iloc[0], eval_exact_match=float((ev[p].round(9) == mode.iloc[0]).mean())))
mb = pd.DataFrame(rows)
if len(mb):
    display(mb.style.format({"train_mode": "{:.3g}", "eval_exact_match": "{:.1%}"}))
    audit("猜众数基线（各材料×参数 eval_exact_match 的均值）", f"{mb.eval_exact_match.mean():.1%}", "PASS", "模型的参数准确率必须明显高于这条基线才算读了视频；见 mode_baseline.csv")
    mb.to_csv(OUT_DIR / "mode_baseline.csv", index=False)

from scipy.stats import ks_2samp
ks = []
for p in ["logrho", "logE", "nu", "logE_over_rho", "sim_dt_per_frame"]:
    a, b = TR[p].dropna(), EV[p].dropna()
    if len(a) > 5 and len(b) > 5: r = ks_2samp(a, b); ks.append(dict(param=p, ks=r.statistic, p=r.pvalue))
display(pd.DataFrame(ks).round(4))
cmt = pd.concat({s: meta[meta.split == s].cm.value_counts(normalize=True) for s in SPLITS}, axis=1).fillna(0).round(3); display(cmt)

## 7. 提示侧泄漏清单（构建 SFT 数据时必须遵守）

In [ ]:
print(f"yaml 含 description 字段的场景占比: {meta.has_desc.mean():.1%}")
print("场景路径/目录名含 'base' / 'perturbation'，且与参数是否被扰动相关：")
print(meta.groupby("kind").agg(n=("scene", "count"), non_mode_E=("E", lambda s: float((s.round(9) != s.round(9).mode().iloc[0]).mean()) if s.notna().any() else np.nan)))
audit("提示侧：description / 路径 / 场景名", "需人工保证", "WARN",
      "SFT 的 prompt 里不得包含 scene 路径、场景名、description、yaml 中的任何标签字段（video-only 主任务）；perturbation 命名本身携带标签信息")

## 8. 汇总

In [ ]:
res = pd.DataFrame(AUDIT)
col_ = {"PASS": "#d4edda", "WARN": "#fff3cd", "FAIL": "#f8d7da"}
sty = res.style; fn = sty.map if hasattr(sty, "map") else sty.applymap
display(fn(lambda v: f"background-color: {col_.get(v, '')}", subset=["status"]))
res.to_csv(OUT_DIR / "audit_summary.csv", index=False)
n_fail, n_warn = int((res.status == "FAIL").sum()), int((res.status == "WARN").sum())
verdict = "不要开始训练：存在 FAIL 项" if n_fail else ("可以训练，但先处理 WARN 项并在报告里注明" if n_warn else "通过")
lines = ["# 数据泄漏审查报告", f"\n结论：**{verdict}**（FAIL {n_fail}，WARN {n_warn}）\n", "| 检查 | 值 | 状态 | 备注 |", "|---|---|---|---|"]
lines += [f"| {r.check} | {r.value} | {r.status} | {r.note} |" for r in res.itertuples()]
(OUT_DIR / "leakage_audit.md").write_text("\n".join(lines))
print("结论:", verdict); print("saved to", OUT_DIR.resolve())